## Fetch Company Tickers

- Our goal currently is to automatically extract company ticker information from the a trusted source i.e [CNBC Dow30](https://www.cnbc.com/dow-30/).
- We devise this action currently by using relevant libraries for extraction from the above seed URL.
- Later after getting the relevant links, we reorganize this data and aggregate the data across three/four properties -- Company Name, Ticker, Homepage URL.
- We validate the output before going to the next phase -- fetching IR URl.

Current Issues

- We are not able to extract information from the website for the primary reason that majority of the websites operate on a Javascript loading basis. So we need a library that simulates that...

> Using playwright as our primary library for broswer based extraction. 

In [ ]:
from playwright.async_api import async_playwright

async def scrape_apple_ir():
    """Scrape Apple Investor Relations page"""
    
    # Create the playwright instance
    async with async_playwright() as p:
        # Launch browser inside the context
        browser = await p.chromium.launch(headless=True)
        
        # Create page inside browser context
        page = await browser.new_page()
        
        try:
            print("🌐 Loading Apple Investor Relations...")
            
            # Navigate to page
            await page.goto('https://investor.apple.com/', wait_until='networkidle', timeout=30000)
            
            print("✓ Page loaded!")
            
            # Wait a bit for dynamic content
            await page.wait_for_timeout(2000)
            
            # Get page title to confirm it loaded
            title = await page.title()
            print(f"📄 Page title: {title}")
            
            # Get all PDF links
            pdf_links = await page.locator('a[href*=".pdf"]').all()
            print(f"📑 Found {len(pdf_links)} PDF links")
            
            # Extract link data
            results = []
            for link in pdf_links[:10]:  # First 10 links
                try:
                    href = await link.get_attribute('href')
                    text = await link.inner_text()
                    results.append({
                        'text': text.strip(),
                        'url': href
                    })
                except:
                    continue
            
            print(f"✓ Extracted {len(results)} links")
            
            return results
            
        except Exception as e:
            print(f"❌ Error: {e}")
            return []
            
        finally:
            # Always close browser
            await browser.close()

# Run the function
results = await scrape_apple_ir()

# Display results
print("\n📋 Results:")
for i, r in enumerate(results, 1):
    print(f"{i}. {r['text'][:60]} -> {r['url']}")

🌐 Loading Apple Investor Relations...
✓ Page loaded!
📄 Page title: Investor Relations - Apple
📑 Found 37 PDF links
✓ Extracted 10 links

📋 Results:
1. Financial Statements -> https://www.apple.com/newsroom/pdfs/fy2025-q3/FY25_Q3_Consolidated_Financial_Statements.pdf
2. 10-Q -> https://s2.q4cdn.com/470004039/files/doc_earnings/2025/q3/filing/10Q-Q3-2025-as-filed.pdf
3. Financial Statements -> https://www.apple.com/newsroom/pdfs/fy2025-q2/FY25_Q2_Consolidated_Financial_Statements.pdf
4. 10-Q -> https://s2.q4cdn.com/470004039/files/doc_earnings/2025/q2/filing/10Q-Q2-2025-as-filed.pdf
5. Financial Statements -> https://www.apple.com/newsroom/pdfs/fy2025-q1/FY25_Q1_Consolidated_Financial_Statements.pdf
6. 10-Q -> https://s2.q4cdn.com/470004039/files/doc_earnings/2025/q1/filing/10Q-Q1-2025-as-filed.pdf
7. Financial Statements -> https://www.apple.com/newsroom/pdfs/fy2024-q4/FY24_Q4_Consolidated_Financial_Statements.pdf
8. 10-K -> https://s2.q4cdn.com/470004039/files/doc_earnings/2024/q4/fili

### Extraction of Tickers and Link Information from CNBC DOW30 Webpage

In [19]:
CNBC_DOW30_URL='https://www.cnbc.com/dow-30/'

In [22]:
#Use playwright to extract the tickers and link information from the CNBC DOW30 Webpage
async def scrape_dow30_tickers():
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        try:
            page = await browser.new_page()
            await page.goto(CNBC_DOW30_URL)
            print("📱 Accessing CNBC DOW30 page...")

            # Wait for the table to load
            await page.wait_for_selector('table')
            
            # Extract ticker information
            ticker_elements = await page.locator('tr').all()
            
            results = []
            for element in ticker_elements[1:]:  # Skip header row
                try:
                    # Get cells and links from each row
                    cells = await element.locator('td').all()
                    
                    # Get any links in the row
                    links = await element.locator('a').all()
                    link_urls = []
                    for link in links:
                        href = await link.get_attribute('href')
                        if href:
                            link_urls.append(href)
                    
                    if len(cells) >= 2:
                        symbol = await cells[0].inner_text()
                        company = await cells[1].inner_text()
                        
                        results.append({
                            'ticker': symbol.strip(),
                            'company_name': company.strip(),
                            'link_urls': link_urls
                        })
                except Exception as e:
                    print(f"Error processing row: {e}")
                    continue
                    
            print(f"✓ Extracted {len(results)} tickers")
            return results

        except Exception as e:
            print(f"❌ Error: {e}")
            return []
            
        finally:
            await browser.close()

# Run the function
dow30_tickers = await scrape_dow30_tickers()

# Display results
print("\n📋 DOW 30 Tickers:")
for ticker in dow30_tickers:
    print(f"{ticker['ticker']}: {ticker['company_name']} {ticker['link_urls']}")



📱 Accessing CNBC DOW30 page...
✓ Extracted 30 tickers

📋 DOW 30 Tickers:
AMGN: Amgen Inc ['//www.cnbc.com/quotes/AMGN']
AMZN: Amazon.com Inc ['//www.cnbc.com/quotes/AMZN']
HON: Honeywell International Inc ['//www.cnbc.com/quotes/HON']
MSFT: Microsoft Corp ['//www.cnbc.com/quotes/MSFT']
NVDA: NVIDIA Corp ['//www.cnbc.com/quotes/NVDA']
KO: Coca-Cola Co ['//www.cnbc.com/quotes/KO']
SHW: Sherwin-Williams Co ['//www.cnbc.com/quotes/SHW']
HD: Home Depot Inc ['//www.cnbc.com/quotes/HD']
IBM: International Business Machines Corp ['//www.cnbc.com/quotes/IBM']
JNJ: Johnson & Johnson ['//www.cnbc.com/quotes/JNJ']
JPM: JPMorgan Chase & Co ['//www.cnbc.com/quotes/JPM']
MCD: McDonald’s Corp ['//www.cnbc.com/quotes/MCD']
MMM: 3M Co ['//www.cnbc.com/quotes/MMM']
MRK: Merck & Co Inc ['//www.cnbc.com/quotes/MRK']
NKE: Nike Inc ['//www.cnbc.com/quotes/NKE']
PG: Procter & Gamble Co ['//www.cnbc.com/quotes/PG']
AXP: American Express Co ['//www.cnbc.com/quotes/AXP']
BA: Boeing Co ['//www.cnbc.com/quotes/BA'